## 1.1直接调用

In [1]:
from langchain_core.tools import tool
@tool
def get_weather(city: str) -> str:
    """
    获取指定城市的天气信息
    参数:
    city: 城市名称，如"北京"、"上海"
    返回:
    天气信息字符串
    """
# 你的实现
    return city + "晴天，温度 15°C"
# 使用 .invoke() 方法
result = get_weather.invoke({"city": "北京"})
print(result)

北京晴天，温度 15°C


### 1.2 基于模型的调用

In [3]:
import os
from dotenv import load_dotenv
from langchain.chat_models import init_chat_model
from langchain_core.tools import tool

load_dotenv(override=True)

DEEPSEEK_API_KEY = os.getenv("DEEPSEEK_API_KEY")
DEEPSEEK_BASE_URL = os.getenv("DEEPSEEK_BASE_URL")

model = init_chat_model(
    model= "deepseek-v4-flash",
    api_key = DEEPSEEK_API_KEY,
    base_url = DEEPSEEK_BASE_URL
)

@tool
def get_weather(city: str) -> str:
    """
    获取指定城市的天气信息
    参数:
    city: 城市名称，如"北京"、"上海"
    返回:
    天气信息字符串
    """
    return "晴天，温度 15°C"


# AI 模型绑定工具
model_with_tools = model.bind_tools([get_weather])

response = model_with_tools.invoke("北京的天气如何？")

if response.tool_calls:
    print("AI 想调用工具：", response.tool_calls)
else:
    print("AI 直接回答：", response.content)

AI 想调用工具： [{'name': 'get_weather', 'args': {'city': '北京'}, 'id': 'call_00_MSGZk53BoFHECt18jPFU9738', 'type': 'tool_call'}]


从message流转看工具的调用

In [4]:
from langchain.messages import HumanMessage, ToolMessage
@tool
def get_weather(city: str):
    """获取天气的工具"""
    return f"{city}天气晴朗~"
# 将模型和工具绑定
model_with_tools = model.bind_tools([get_weather])
messages = [
    HumanMessage("今天北京天气如何")
]

# 模型生成调用工具请求
response = model_with_tools.invoke(messages)

# 添加AIMessage
messages.append(response)
tool_calls = response.tool_calls
for tool_call in tool_calls:
    if tool_call["name"] == "get_weather":
        # 返回的是ToolMessage类型消息,添加到消息列表
        tool_response = get_weather.invoke(tool_call)
        print(type(tool_response))
        messages.append(tool_response)
print("=====================> messages <=====================")
for msg in messages:
    msg.pretty_print()
print("=====================> messages <=====================")
final_response = model_with_tools.invoke(messages)
print(f"final_response: \n{final_response}")

<class 'langchain_core.messages.tool.ToolMessage'>
=====================> messages <=====================
================================ Human Message =================================

今天北京天气如何
================================== Ai Message ==================================
Tool Calls:
  get_weather (call_00_2XEEfEHZ5tl8Pfis9ub23777)
 Call ID: call_00_2XEEfEHZ5tl8Pfis9ub23777
  Args:
    city: 北京
================================= Tool Message =================================
Name: get_weather

北京天气晴朗~
=====================> messages <=====================
final_response: 
content='北京今天天气晴朗☀️，很适合出门活动哦～' additional_kwargs={'refusal': None, 'reasoning_content': ''} response_metadata={'token_usage': {'completion_tokens': 14, 'prompt_tokens': 427, 'total_tokens': 441, 'completion_tokens_details': {'accepted_prediction_tokens': None, 'audio_tokens': None, 'reasoning_tokens': 0, 'rejected_prediction_tokens': None}, 'prompt_tokens_details': {'audio_tokens': None, 'cache_write_tokens': None